# 🚗 MrCrayfish Vehicle Plugin — server de test pe Colab

Rulează un server **Paper 1.21.4** cu **MrCrayfishVehiclePlugin** direct din acest notebook, fără nicio gazdă proprie. Publicat pe internet prin **un tunel FRP** (relay-ul tău `frps`) sau **playit.gg**.

> ⚠️ **Citește înainte de prima rulare**
> 1. **Termenii Google Colab interzic găzduirea de servere de jocuri.** Riscul real este restricționarea accesului la Colab pentru contul folosit. Recomandarea noastră: folosește un **cont Google secundar**, nu contul principal.
> 2. Runtime-ul gratuit: maxim **~12 ore** per sesiune, deconectare la **~90 min de inactivitate**, iar fila trebuie să rămână deschisă. La fiecare reconectare totul se reinstalează (celulele se re-rulează în ordine), lumea se restaurează din Google Drive.
> 3. Mașinile Colab sunt de obicei în **SUA** → latență ~120–180 ms din România. Suficient pentru teste, nu pentru competiție.
> 4. Tunelul expune serverul pe internet public. **Whitelist-ul e activat implicit** — doar jucătorii din `PLAYER_NAMES` pot intra. Nu dezactiva `online-mode`.
> 5. Nu salva notebook-ul cu token-ul FRP vizibil într-un loc public. Ideal: pune token-ul în **Colab Secrets** (iconița 🔑 din stânga) cu numele `FRPS_TOKEN`, celula de configurare îl citește automat de acolo.

**Ordinea de rulare:** celulele de sus în jos (sau *Runtime → Run all*). Dacă runtime-ul se resetează, rulează-le din nou de la început — lumea revine automat din Drive.

In [ ]:
#@title ⚙️ Configurare — completează și rulează celula { display-mode: "form" }
import json, os

#@markdown ### Minecraft / plugin
MC_VERSION = "1.21.4" #@param {type:"string"}
RELEASE_TAG = "vehicle-plugin-prototype-v0.1.0" #@param {type:"string"}
RAM = "8G" #@param ["4G", "6G", "8G", "10G"]

#@markdown ### Tunel public
TUNNEL = "frp" #@param ["frp", "playit"]
FRPS_ADDR = "" #@param {type:"string"}
FRPS_PORT = 7000 #@param {type:"integer"}
FRPS_REMOTE_PORT = 25565 #@param {type:"integer"}
FRPS_TOKEN = "" #@param {type:"string"}

#@markdown ### Jucători (nume Minecraft, separate prin virgulă). Primesc automat OP + whitelist.
PLAYER_NAMES = "" #@param {type:"string"}

#@markdown ### Persistență
PERSIST_TO_DRIVE = True #@param {type:"boolean"}

# Tokenul poate veni din Colab Secrets (recomandat).
if not FRPS_TOKEN:
    try:
        from google.colab import userdata
        FRPS_TOKEN = userdata.get("FRPS_TOKEN") or ""
    except Exception:
        pass

MC_DIR = "/content/mc"
os.makedirs(MC_DIR, exist_ok=True)

ops = " ".join(n.strip() for n in PLAYER_NAMES.split(",") if n.strip())

with open(f"{MC_DIR}/env.sh", "w") as f:
    f.write(f'MC_VERSION="{MC_VERSION}"\n')
    f.write(f'RELEASE_TAG="{RELEASE_TAG}"\n')
    f.write(f'RAM="{RAM}"\n')
    f.write(f'TUNNEL="{TUNNEL}"\n')
    f.write(f'FRPS_ADDR="{FRPS_ADDR}"\n')
    f.write(f'FRPS_PORT="{FRPS_PORT}"\n')
    f.write(f'FRPS_REMOTE_PORT="{FRPS_REMOTE_PORT}"\n')
    f.write(f'FRPS_TOKEN="{FRPS_TOKEN}"\n')
    f.write(f'OP_LIST="{ops}"\n')
    f.write(f'MC_DIR="{MC_DIR}"\n')

print("✅ Configurație salvată.")
print(f"   Tunel: {TUNNEL}", end="")
if TUNNEL == "frp":
    assert FRPS_ADDR, "❌ TUNNEL=frp dar FRPS_ADDR e gol! Completează adresa serverului tău frps."
    assert FRPS_TOKEN, "❌ FRPS_TOKEN e gol. Pune-l în celulă sau în Colab Secrets."
    print(f" → {FRPS_ADDR}:{FRPS_REMOTE_PORT}")
else:
    print(" → linkul de claim playit va apărea la celula de tunel")
print(f"   Jucători cu OP: {ops or '(niciunul — completează PLAYER_NAMES!)'}")
print(f"   RAM: {RAM}, Minecraft {MC_VERSION}, release {RELEASE_TAG}")

## 1. Java 21
Paper 1.21.4 cere Java 21. Se instalează prin `apt` (durează ~1 min).

In [ ]:
%%bash
set -e
if /usr/lib/jvm/java-21-openjdk-amd64/bin/java -version 2>/dev/null; then
  echo "Java 21 deja instalat."
else
  apt-get update -qq
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-21-jre-headless >/dev/null
fi
/usr/lib/jvm/java-21-openjdk-amd64/bin/java -version

## 2. Serverul Paper
Descarcă cel mai recent build Paper pentru versiunea aleasă.

In [ ]:
%%bash
set -euo pipefail
source /content/mc/env.sh
cd "$MC_DIR"
echo "→ Caut ultimul build Paper $MC_VERSION…"
BUILDS=$(curl -fsSL "https://api.papermc.io/v2/projects/paper/versions/$MC_VERSION/builds")
BUILD=$(echo "$BUILDS" | python3 -c 'import sys,json; print(json.load(sys.stdin)["builds"][-1]["build"])')
JAR=$(echo "$BUILDS" | python3 -c 'import sys,json; print(json.load(sys.stdin)["builds"][-1]["downloads"]["application"]["name"])')
SHA256=$(echo "$BUILDS" | python3 -c 'import sys,json; print(json.load(sys.stdin)["builds"][-1]["downloads"]["application"]["sha256"])')
[ -f paper.jar ] || curl -fsSL -o paper.jar "https://api.papermc.io/v2/projects/paper/versions/$MC_VERSION/builds/$BUILD/downloads/$JAR"
echo "$SHA256  paper.jar" | sha256sum -c -
ls -la paper.jar

## 3. Pluginul + resource pack-ul
Descarcă JAR-ul și **cel mai recent pack** din release-ul GitHub ales și calculează SHA-1-ul real al pack-ului (configul pluginului se generează din el, deci nu depindem de valori hardcodate).

In [ ]:
%%bash
set -euo pipefail
source /content/mc/env.sh
cd "$MC_DIR"
mkdir -p plugins
REPO="ExMarius/MrCrayfishVehicleMod"
BASE="https://github.com/$REPO/releases/download/$RELEASE_TAG"

ASSETS=$(curl -fsSL "https://api.github.com/repos/$REPO/releases/tags/$RELEASE_TAG")
JAR_ASSET=$(echo "$ASSETS" | python3 -c 'import sys,json; print([a["name"] for a in json.load(sys.stdin)["assets"] if a["name"].endswith(".jar")][0])')
[ -f plugins/MrCrayfishVehiclePlugin.jar ] || curl -fsSL -o plugins/MrCrayfishVehiclePlugin.jar "$BASE/$JAR_ASSET"
echo "✅ Plugin: $JAR_ASSET"

# Cel mai recent pack (cel mai mare sufix rNN).
echo "$ASSETS" | python3 > /tmp/pack_name.txt -c '
import re, sys, json
names = [a["name"] for a in json.load(sys.stdin)["assets"]]
cands = [n for n in names if re.search(r"-r\d+\.zip$", n)]
best = max(cands, key=lambda n: int(re.search(r"-r(\d+)\.zip$", n).group(1)))
print(best)'
PACK=$(cat /tmp/pack_name.txt)
curl -fsSL -o "$PACK" "$BASE/$PACK"
echo "$PACK  →  $(sha1sum "$PACK")"
{
  echo "PACK_NAME="$PACK""
  echo "PACK_URL="$BASE/$PACK""
  echo "PACK_SHA1="$(sha1sum "$PACK" | awk '{print $1}')""
} >> "$MC_DIR/env.sh"
ls -la plugins/ "$PACK" 

## 4. Fișierele serverului
`server.properties` e reglat pentru Colab: view/simulation distance modeste, `allow-flight=true` (**obligatoriu** pentru vehicule), whitelist activ, spawn-protection oprit.

In [ ]:
%%bash
set -euo pipefail
source /content/mc/env.sh
cd "$MC_DIR"
echo "eula=true" > eula.txt
cat > server.properties <<'EOF'
motd=MrCrayfish Vehicle Test (Colab)
server-port=25565
online-mode=true
white-list=true
enforce-whitelist=true
allow-flight=true
spawn-protection=0
max-players=8
view-distance=6
simulation-distance=4
sync-chunk-writes=false
difficulty=peaceful
spawn-monsters=false
level-type=minecraft:flat
generate-structures=false
EOF
echo "✅ eula.txt + server.properties scrise (lume superflat, peaceful — pistol de start pentru teste)."
echo "   Schimbă level-type dacă vrei alt teren."


## 5. Restaurare din Google Drive
Montează Drive și aduce înapoi lumea + datele pluginului (vehicule parcate, combustibil) salvate la sesiunea anterioară. La prima rulare nu există nimic de restaurat — e normal.

In [ ]:
import os, shutil

MC_DIR = "/content/mc"
BACKUP = "/content/drive/MyDrive/mcv-backup"
PERSIST = True  # setat de celula de configurare (vezi mai jos)

try:
    with open(f"{MC_DIR}/env.sh") as f:
        env = dict(l.strip().split("=", 1) for l in f if "=" in l)
except FileNotFoundError:
    env = {}

# PERSIST_TO_DRIVE din celula de configurare e recitit aici manual la re-rulare;
# implicit True. Nu monta Drive dacă utilizatorul a oprit persistența.
from google.colab import drive  # doar în Colab
drive.mount("/content/drive")

restored = []
for entry in ["world", "world_nether", "world_the_end",
              "plugins/MrCrayfishVehiclePlugin", "banned-players.json",
              "whitelist.json", "ops.json"]:
    src = os.path.join(BACKUP, entry)
    dst = os.path.join(MC_DIR, entry)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.isdir(src):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        restored.append(entry)

print("✅ Restaurat din Drive:" if restored else "ℹ️  Nimic de restaurat (prima rulare).", ", ".join(restored))

## 6. Configul pluginului
Generat **după** restaurare (ca să suprascrie configul vechi din backup) cu URL-ul și SHA-1-ul real al pack-ului.

In [ ]:
%%bash
set -euo pipefail
source /content/mc/env.sh
cd "$MC_DIR"
mkdir -p plugins/MrCrayfishVehiclePlugin
cat > plugins/MrCrayfishVehiclePlugin/config.yml <<EOF
# Generat automat de notebook-ul Colab — nu edita manual.
resource-pack:
  url: "$PACK_URL"
  sha1: "$PACK_SHA1"
  required: true
  prompt: "Acest resource pack este necesar pentru modelele și sunetele vehiculelor."

physics:
  global-speed-limit: 100.0
  fuel-consumption-factor: 1.0

performance:
  activation-distance: 64.0

gas-pump:
  max-hose-distance: 10.0
  fill-seconds: 15.0

persistence:
  autosave-seconds: 300
EOF
echo "✅ config.yml generat pentru $PACK_NAME (sha1: ${PACK_SHA1:0:10}…)"


## 7. Tunelul public

**Varianta A — FRP:** descarcă `frpc`, îl pornește cu supervisor (reconectare automată la fiecare 15s, ca pe fosta gazdă Oracle). Necesită serverul tău `frps` cu portul `FRPS_REMOTE_PORT` liber.

**Varianta B — playit.gg:** descarcă agentul; la **prima** rulare afișează un link „claim" — deschide-l în browser, fă cont gratuit și revendică agentul; tunelul primește apoi o adresă publică stabilă (ex. `xyz.at.ply.gg`).

In [ ]:
%%bash
set -euo pipefail
source /content/mc/env.sh
cd "$MC_DIR"

if [ "$TUNNEL" = "frp" ]; then
  if [ ! -x /usr/local/bin/frpc ]; then
    TAG=$(curl -fsSL https://api.github.com/repos/fatedier/frp/releases/latest | python3 -c 'import sys,json; print(json.load(sys.stdin)["tag_name"])')
    curl -fsSL -o frp.tgz "https://github.com/fatedier/frp/releases/download/$TAG/frp_${TAG#v}_linux_amd64.tar.gz"
    tar xzf frp.tgz
    cp "frp_${TAG#v}_linux_amd64/frpc" /usr/local/bin/frpc
    chmod +x /usr/local/bin/frpc
  fi
  cat > frpc.toml <<EOF
serverAddr = "$FRPS_ADDR"
serverPort = $FRPS_PORT
auth.method = "token"
auth.token = "$FRPS_TOKEN"

[[proxies]]
name = "minecraft"
type = "tcp"
localIP = "127.0.0.1"
localPort = 25565
remotePort = $FRPS_REMOTE_PORT
EOF
  pkill -x frpc 2>/dev/null || true
  nohup bash -c 'while true; do frpc -c /content/mc/frpc.toml >> /content/mc/frpc.log 2>&1; sleep 15; done' >/dev/null 2>&1 &
  sleep 4
  tail -n 6 frpc.log || true
  echo
  echo "════════════════════════════════════════════════════════"
  echo "🟢 Adresa de conectare:  $FRPS_ADDR:$FRPS_REMOTE_PORT"
  echo "════════════════════════════════════════════════════════"
else
  if [ ! -x ./playit ]; then
    curl -fsSL -o playit https://github.com/playit-cloud/playit-agent/releases/latest/download/playit-linux
    chmod +x playit
  fi
  pkill -x playit 2>/dev/null || true
  nohup ./playit > playit.log 2>&1 &
  sleep 6
  grep -Eo 'https://playit.gg/claim/[A-Za-z0-9_-]+' playit.log | head -1 | while read -r url; do
    echo "👉 PRIMA RULARE: deschide și revendică agentul: $url"
  done
  grep -Eo '[a-z0-9-]+\.at\.ply\.gg(:[0-9]+)?' playit.log | head -1 | while read -r addr; do
    echo "🟢 Adresa de conectare:  $addr"
  done
  echo "(jurnal: tail -f /content/mc/playit.log)"
fi

## 8. Pornirea serverului
Pornește Paper în fundal (intrare prin `console.in`, jurnal în `console.log`), așteaptă „Done", apoi op-ează și whitelist-uiește automat jucătorii configurați. Durează ~2–4 min la prima pornire (se generează lumea).

In [ ]:
%%bash
set -uo pipefail
source /content/mc/env.sh
cd "$MC_DIR"
JAVA=/usr/lib/jvm/java-21-openjdk-amd64/bin/java

if pgrep -f "paper.jar" >/dev/null; then
  echo "Serverul rulează deja. Oprește-l mai întâi din celula de oprire."
  exit 0
fi

[ -p console.in ] || mkfifo console.in
: > console.log
nohup bash -c "
  tail -n +1 -F console.in | $JAVA -Xms2G -Xmx$RAM     -XX:+UseG1GC -XX:+ParallelRefProcEnabled -XX:MaxGCPauseMillis=200     -jar paper.jar nogui >> console.log 2>&1
" >/dev/null 2>&1 &

echo "→ Se pornește…"
for i in $(seq 1 60); do
  if grep -q 'Done (' console.log 2>/dev/null; then break; fi
  if ! pgrep -f "paper.jar" >/dev/null; then
    echo "❌ Serverul s-a oprit. Ultimele linii:"; tail -n 30 console.log; exit 1
  fi
  sleep 5
done
grep -q 'Done (' console.log || { echo "❌ Timeout la pornire:"; tail -n 30 console.log; exit 1; }
grep 'Done (' console.log | tail -1

for name in $OP_LIST; do
  echo "whitelist add $name" > console.in
  echo "op $name" > console.in
  echo "→ OP + whitelist: $name"
done
sleep 2
tail -n 5 console.log

## 9. Consolă
Trimite o comandă către server (ex. `vehicle spawn go_kart`, `list`, `say Salut`). Scrie comanda în câmp și rulează celula.

In [ ]:
%%bash
#@title 💬 Trimite o comandă { display-mode: "form" }
CMD = "list" #@param {type:"string"}
cd /content/mc
echo "$CMD" > console.in
sleep 2
tail -n 12 console.log

## 10. Stare
Verifică serverul, tunelul și RULAREA curentă.

In [ ]:
%%bash
source /content/mc/env.sh
echo "── Procese:"
pgrep -af "paper.jar" || echo "  server: OPRIT"
pgrep -x frpc >/dev/null && echo "  frpc: rulează" || true
pgrep -x playit >/dev/null && echo "  playit: rulează" || true
echo
echo "── Memorie:"; free -h | head -2
echo
echo "── Ultimele linii din consolă:"
tail -n 15 /content/mc/console.log 2>/dev/null || echo "(fără jurnal)"
if [ "$TUNNEL" = "frp" ]; then
  echo
  echo "── Adresă publică: $FRPS_ADDR:$FRPS_REMOTE_PORT"
else
  echo
  grep -Eo '[a-z0-9-]+\.at\.ply\.gg(:[0-9]+)?' /content/mc/playit.log 2>/dev/null | head -1 | while read -r a; do echo "── Adresă publică: $a"; done
fi

## 11. Oprire + backup în Drive
Oprește serverul **grațios** (`stop` — salvează lumea), apoi sincronizează lumea și datele pluginului în `MyDrive/mcv-backup/`. Rulează mereu această celulă **înainte** să închizi sau să lași runtime-ul să expire, altfel pierzi progresul de la ultimul backup.

In [ ]:
%%bash
source /content/mc/env.sh
cd "$MC_DIR"
if pgrep -f "paper.jar" >/dev/null; then
  echo "→ Trimit 'stop'…"
  echo "stop" > console.in
  for i in $(seq 1 24); do
    pgrep -f "paper.jar" >/dev/null || break
    sleep 5
  done
  pgrep -f "paper.jar" >/dev/null && { echo "⚠️ Forțez oprirea"; pkill -f "paper.jar"; sleep 3; } || echo "✅ Server oprit curat."
else
  echo "Serverul nu rulează."
fi

In [ ]:
import os, shutil

MC_DIR = "/content/mc"
BACKUP = "/content/drive/MyDrive/mcv-backup"

from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
os.makedirs(BACKUP, exist_ok=True)

saved = []
for entry in ["world", "world_nether", "world_the_end",
              "plugins/MrCrayfishVehiclePlugin", "whitelist.json", "ops.json"]:
    src = os.path.join(MC_DIR, entry)
    dst = os.path.join(BACKUP, entry)
    if os.path.exists(src):
        if os.path.isdir(src):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        saved.append(entry)
print("✅ Backup în Drive:", ", ".join(saved) if saved else "nimic de salvat")
import subprocess
print(subprocess.run(["du", "-sh", BACKUP], capture_output=True, text=True).stdout.strip())

---
## Depanare rapidă

| Simptom | Cauză probabilă | Remediu |
|---|---|---|
| `FRPS_ADDR e gol` | config incomplet | celula de configurare → adresa frps |
| frpc: `connection refused` / timeout | frps oprit sau port firewall | verifică că `frps` rulează și `bindPort` (7000) + portul remote (25565) sunt deschise |
| frpc: `authentication failed` | token greșit | sincronizează `auth.token` cu `frps.toml` |
| playit fără adresă | agent nerevendicat | deschide link-ul de claim din jurnal |
| „Flying is not enabled" la vehicule | `allow-flight=false` | celula 4 rescrie `server.properties` — ruleaz-o din nou |
| Client refuză pack-ul | sha1/url vechi | re-rulează celulele 3 și 6 (config regenerat) și repornește serverul |
| Lag / TPS mic | runtime Colab slab | `RAM` mai mare nu ajută CPU-ul; redu view-distance în celula 4 |
| Runtime deconectat | cap de 12h / idle | re-rulează tot; lumea revine din Drive |

**Actualizarea pluginului:** când apare un release nou, schimbă `RELEASE_TAG` în celula de configurare și re-rulează celulele 3 → 8 (serverul trebuie oprit între timp).

**Securitate:** nu da adresa tunelului necunoscuților; whitelist=ON; nu publica `frpc.toml` (conține tokenul).